# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meer140/flyrank-ml-internship-project/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Provisional Lane Selection:**
**Lane 2: Refresh / Content Opportunity Scoring**

**Why this lane?**
Managing content portfolios across multi-client digital ecosystems presents a significant resource allocation challenge: content strategists and editors have limited bandwidth, while published content continuously decays, loses search rank, or faces shifting organic search demand over time. Rather than guessing which pages to refresh or relying on blunt single-dimension rules that flood teams with thousands of unprioritized tasks, **Lane 2** directly addresses **opportunistic content refresh scoring**. It focuses on identifying, scoring, and ranking published pages that have demonstrated search volume but are experiencing traffic decay or underperforming their potential, ensuring editorial effort yields the highest return on investment.

In [1]:
# Check baseline availability for Lane 2
print("Lane 2 selected: Refresh / Content Opportunity Scoring")
print("Focus: Prioritizing content refresh, expansion, protection, and monitoring queues.")

Lane 2 selected: Refresh / Content Opportunity Scoring
Focus: Prioritizing content refresh, expansion, protection, and monitoring queues.


## 2. The question: decision, action, cost of a wrong call

### 1. Research Question & Unit of Analysis
* **Search Question:** *Which high-demand content items across client portfolios should be reviewed and refreshed first to halt visibility decay and recapture organic search traffic?*
* **Unit of Analysis:** A pseudonymized content item / page over a trailing 90-day performance window (`content_id` grain per `client_id`).

### 2. The Decision
* **What decision does this work improve?** Deciding how to allocate limited editorial capacity across thousands of published pages — replacing ad-hoc manual checks or blunt unweighted threshold rules with an evidence-backed, prioritized review queue.

### 3. Who Acts & What Action Is Taken
* **Actor:** Content Editor, SEO Specialist, or Content Strategist.
* **Action:** Inspect the top-ranked review queue (e.g. top 20–50 candidate pages per month), review transparent reason codes (`declining_with_demand`, `low_ctr_visible_page`, `page_one_decay_risk`), and execute targeted editorial updates (content refreshes, structural expansions, title/meta snippet rewrites, or protection of declining cornerstone content).

### 4. Cost of a Wrong Recommendation
* **False Positive (Flagging a healthy page for refresh):** Wastes scarce editorial hours (approx. 2–4 hours of expert writing/editing per page) updating content that was performing well or experiencing temporary seasonal fluctuations. Unnecessary changes can also inadvertently disrupt existing search ranking signals.
* **False Negative (Missing a high-demand page in active decay):** Leads to compounding loss of organic search impressions and clicks, uncaptured revenue, and loss of competitive search positions before manual detection occurs.

### 5. Why Data or ML Helps (Why a plain rule isn't enough)
A plain static rule (such as `if impressions >= 500 and trend == 'down'`) is far too crude: in our dataset, it flags nearly **10,000 pages** simultaneously without accounting for relative search volume, position decay rate, engagement signals, or content age. No editorial team can manually evaluate 10,000 pages. Machine learning combines non-linear, multi-dimensional signals into a unified, calibrated probability/score that ranks candidates by true opportunity and risk.

In [2]:
# Frame verification check
unit_of_analysis = "Content Item / Page (content_id grain over trailing 90 days)"
target_type = "Refresh Priority / Decline Outcome"
metric = "Precision@K (e.g. Precision@50) & Average Precision"

print(f"Unit of Analysis: {unit_of_analysis}")
print(f"Target Type: {target_type}")
print(f"Primary Metric: {metric}")

Unit of Analysis: Content Item / Page (content_id grain over trailing 90 days)
Target Type: Refresh Priority / Decline Outcome
Primary Metric: Precision@K (e.g. Precision@50) & Average Precision


## 3. Quick look at the data (2-3 real numbers)

To verify that Lane 2 is worth our next 7 weeks, we loaded the starter anonymized dataset (`data/raw/content_refresh_anonymized.csv`), which contains **30,000 pseudonymized content items across 32 clients**. The empirical numbers directly support our problem framing:

1. **Widespread Content Decay:** Out of 30,000 total content items, **16,262 pages (54.2%)** exhibit an active downward performance trend (`trend_direction == 'down'`).
2. **The Prioritization Problem (High-Demand Decay Pool):** **9,961 pages (33.2%)** combine high search volume (`impressions_90d >= 500`) with an active downward trend. This confirms that simple filtering leaves an unmanageably large queue (~10,000 pages), proving why ML-based opportunity ranking is required.
3. **High-Visibility CTR Optimization Opportunity:** **9,759 pages** sit on page 1 or 2 of search results (`1 <= avg_position <= 20`) with strong visibility (`impressions_90d >= 500`), yet suffer from click-through rates below 0.5% (`ctr < 0.5%`). This highlights a major secondary opportunity for snippet/title optimization alongside content refreshes.

In [3]:
import os
import pandas as pd
import numpy as np

# Robust path handling for Colab, root, or local work/notebooks directory execution
possible_paths = [
    'data/raw/content_refresh_anonymized.csv',
    '../data/raw/content_refresh_anonymized.csv',
    '../../data/raw/content_refresh_anonymized.csv'
]

data_path = None
for p in possible_paths:
    if os.path.exists(p):
        data_path = p
        break

if data_path is None:
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv")

# Load starter dataset
df = pd.read_csv(data_path)

# Calculate key empirical metrics
total_items = len(df)
num_clients = df['client_id'].nunique()
declining_items = (df['trend_direction'] == 'down').sum()
declining_pct = (declining_items / total_items) * 100

high_demand_declining = df[(df['impressions_90d'] >= 500) & (df['trend_direction'] == 'down')].shape[0]
high_demand_declining_pct = (high_demand_declining / total_items) * 100

low_ctr_visible = df[(df['impressions_90d'] >= 500) & (df['avg_position'] > 0) & (df['avg_position'] <= 20) & (df['ctr'] < 0.5)].shape[0]

# Display evidence summary table
summary_df = pd.DataFrame({
    "Metric Description": [
        "Total Content Items (Dataset Scope)",
        "Pseudonymized Clients",
        "Pages with Downward Trend (trend_direction == 'down')",
        "High-Demand Declining Pages (impressions >= 500 & trend == 'down')",
        "Visible Pages with Low CTR (impressions >= 500, pos 1-20, CTR < 0.5%)"
    ],
    "Count / Value": [
        f"{total_items:,}",
        f"{num_clients}",
        f"{declining_items:,}",
        f"{high_demand_declining:,}",
        f"{low_ctr_visible:,}"
    ],
    "Percentage of Dataset": [
        "100.0%",
        "-",
        f"{declining_pct:.1f}%",
        f"{high_demand_declining_pct:.1f}%",
        f"{(low_ctr_visible / total_items)*100:.1f}%"
    ]
})

print("=== FLYRANK STARTER DATASET EVIDENCE (Lane 2 Baseline Support) ===")
print(summary_df.to_string(index=False))

=== FLYRANK STARTER DATASET EVIDENCE (Lane 2 Baseline Support) ===
                                                   Metric Description Count / Value Percentage of Dataset
                                  Total Content Items (Dataset Scope)        30,000                100.0%
                                                Pseudonymized Clients            32                     -
                Pages with Downward Trend (trend_direction == 'down')        16,262                 54.2%
   High-Demand Declining Pages (impressions >= 500 & trend == 'down')         9,961                 33.2%
Visible Pages with Low CTR (impressions >= 500, pos 1-20, CTR < 0.5%)         9,759                 32.5%


## 4. Careful words: what I can and can't claim

### What this work CAN claim:
* **Observational Signal Patterns:** Empirical correlations and associations between observed search/content signals (e.g. word count, age tier, search position, CTR) and historical traffic movement across client portfolios.
* **Empirical Ranking Lift:** Demonstrable performance improvement of a learned scoring model over static hand-written rules, evaluated using decision-centric metrics (Precision@50, Average Precision, ROC-AUC).
* **Decision Support & Action Prioritization:** Prioritized candidate queues with transparent, inspectable reason codes to guide human editorial workflows.

### What this work CANNOT and will NEVER claim:
* **Causal Proof of Traffic Recovery:** We cannot claim that executing a content refresh *causes* a guaranteed traffic recovery; establishing causality requires controlled A/B experiments beyond observational snapshot data.
* **Reverse-Engineering Google's Algorithm:** We do not claim to predict or reverse-engineer Google's internal ranking algorithm; we analyze historical patterns in client search performance.
* **Unmasked Client Identities or URLs:** No claims regarding specific real-world client names, real URLs, or raw search queries — all data remains pseudonymized and public-safe.

In [4]:
# Audit language check
claim_scope = {
    "Allowed Claims": ["Observational correlation", "Ranking precision lift", "Decision support queue"],
    "Forbidden Claims": ["Causal proof of recovery", "Predicting Google algorithm", "Unmasked client identities"]
}

for claim_type, claims in claim_scope.items():
    print(f"{claim_type}:")
    for c in claims:
        print(f"  - {c}")

Allowed Claims:
  - Observational correlation
  - Ranking precision lift
  - Decision support queue
Forbidden Claims:
  - Causal proof of recovery
  - Predicting Google algorithm
  - Unmasked client identities


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.